### Reproducibility Note

This notebook uses relative project paths instead of local absolute paths. The raw Hillstrom CSV file is expected to be included in the project repository as `hillstrom_email_data.csv`. The notebook automatically searches upward from the current working directory until it finds the project CSV file, which allows the notebook to run on different machines after the repository is cloned.

All processed modeling files are saved to `data/processed/`, including `X_train`, `X_test`, `y_train`, `y_test`, `treatment_train`, and `treatment_test`. This ensures that all later CausalML models use the same feature matrix, treatment encoding, outcome variable, and train/test split.

In [4]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 100)

In [5]:
def find_project_root(start_path=None, data_file="hillstrom_email_data.csv"):
    """
    Find the project root by searching upward from the current working directory
    until the raw Hillstrom CSV file is found.
    """
    if start_path is None:
        start_path = Path.cwd()
    else:
        start_path = Path(start_path)

    start_path = start_path.resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / data_file).exists():
            return path

    raise FileNotFoundError(
        f"Could not find {data_file}. "
        "Make sure the CSV is committed to the repo and that you are running this notebook from within the project."
    )


PROJECT_ROOT = find_project_root()

DATA_PATH = PROJECT_ROOT / "hillstrom_email_data.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Processed output directory:", PROCESSED_DIR)

Project root: /Users/chris/Library/CloudStorage/OneDrive-Personal/Rice University MDS/COMP 653 Statistical Machine Learning/Uplift Modeling Semester Project/hillstrom-email-uplift-modeling
Data path: /Users/chris/Library/CloudStorage/OneDrive-Personal/Rice University MDS/COMP 653 Statistical Machine Learning/Uplift Modeling Semester Project/hillstrom-email-uplift-modeling/hillstrom_email_data.csv
Processed output directory: /Users/chris/Library/CloudStorage/OneDrive-Personal/Rice University MDS/COMP 653 Statistical Machine Learning/Uplift Modeling Semester Project/hillstrom-email-uplift-modeling/data/processed


### Load the raw Hillstrom dataset

This cell loads the raw Hillstrom email dataset from the reproducible project path defined earlier. The shape and first few rows are displayed to confirm that the file loaded correctly.

In [6]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


### Standardize column names and clean categorical values

This cell standardizes column names so they are easier to reference in code. It also fixes the `Surburban` spelling in `zip_code` so that suburban customers are represented consistently before one-hot encoding.

In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df["zip_code"] = df["zip_code"].replace({
    "Surburban": "Suburban"
})

df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Suburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Suburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


### Check key categorical variables

This cell checks the values in the main categorical variables. This helps confirm that the treatment groups, zip code categories, and purchase channels are formatted consistently before modeling.

In [8]:
print("Zip code categories:")
print(df["zip_code"].value_counts())

print("\nTreatment groups:")
print(df["segment"].value_counts())

print("\nChannel categories:")
print(df["channel"].value_counts())

Zip code categories:
zip_code
Suburban    28776
Urban       25661
Rural        9563
Name: count, dtype: int64

Treatment groups:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Channel categories:
channel
Web             28217
Phone           28021
Multichannel     7762
Name: count, dtype: int64


### Run basic data quality checks

This cell checks for missing values, duplicate rows, and the final list of column names. These checks help confirm that the dataset is ready for feature engineering.

In [9]:
print("Missing values:")
display(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nColumn names:")
print(df.columns.tolist())

Missing values:


recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64


Duplicate rows:
6562

Column names:
['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


### Duplicate row interpretation

The dataset contains some exact duplicate rows. These are not removed because the dataset does not include a unique customer ID, and multiple customers can naturally share the same feature values, treatment assignment, and outcome values. Removing these rows could incorrectly reduce the sample size and distort the randomized treatment-control comparison.

### Create binary treatment encoding

This cell creates the binary treatment variable for the first-stage uplift modeling setup. Customers who received either the men's email or women's email are coded as treated with `treatment_binary = 1`. Customers who received no email are coded as the control group with `treatment_binary = 0`.

In [11]:
treatment_map_binary = {
    "No E-Mail": 0,
    "Mens E-Mail": 1,
    "Womens E-Mail": 1
}

df["treatment_binary"] = df["segment"].map(treatment_map_binary)

df["treatment_label"] = np.where(
    df["treatment_binary"] == 1,
    "Email",
    "No Email"
)

display(
    df[["segment", "treatment_binary", "treatment_label"]]
    .drop_duplicates()
    .sort_values(["treatment_binary", "segment"])
)

print("Missing treatment encodings:", df["treatment_binary"].isna().sum())

,segment,treatment_binary,treatment_label
1,No E-Mail,0,No Email
3,Mens E-Mail,1,Email
0,Womens E-Mail,1,Email


Missing treatment encodings: 0


### Define model features, outcome, and treatment variables

This cell defines the clean modeling inputs. The feature matrix uses only pre-treatment customer attributes. The primary outcome is `visit`, while `conversion` and `spend` are saved separately as secondary outcomes for later extensions.

In [12]:
feature_cols = [
    "recency",
    "history",
    "history_segment",
    "mens",
    "womens",
    "newbie",
    "zip_code",
    "channel"
]

primary_outcome = "visit"

secondary_outcomes = [
    "conversion",
    "spend"
]

treatment_col = "treatment_binary"

X_raw = df[feature_cols].copy()
y = df[primary_outcome].copy()
secondary_y = df[secondary_outcomes].copy()
treatment = df[treatment_col].copy()
segment_original = df["segment"].copy()

### Check for outcome leakage

This cell confirms that post-campaign variables are not included in the feature matrix. The model features should not include `visit`, `conversion`, `spend`, `segment`, or `treatment_binary`.

In [13]:
leakage_cols = ["visit", "conversion", "spend", "segment", "treatment_binary"]

set(leakage_cols).intersection(X_raw.columns)

set()

### Create one train/test split for all models

This cell creates one shared train/test split that all later models will use. The split is stratified by original treatment group and visit outcome so that the train and test sets have similar treatment and outcome distributions.

In [14]:
stratify_col = df["segment"].astype(str) + "_" + df["visit"].astype(str)

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.25,
    random_state=42,
    stratify=stratify_col
)

X_train_raw = X_raw.loc[train_idx].copy()
X_test_raw = X_raw.loc[test_idx].copy()

y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

treatment_train = treatment.loc[train_idx].copy()
treatment_test = treatment.loc[test_idx].copy()

secondary_y_train = secondary_y.loc[train_idx].copy()
secondary_y_test = secondary_y.loc[test_idx].copy()

segment_train = segment_original.loc[train_idx].copy()
segment_test = segment_original.loc[test_idx].copy()

### Check train/test balance

This cell checks whether the train and test sets have similar treatment group distributions and visit rates. This confirms that the shared split is appropriate for comparing models fairly.

In [15]:
print("Train size:", X_train_raw.shape)
print("Test size:", X_test_raw.shape)

print("\nOriginal treatment balance - train:")
print(segment_train.value_counts(normalize=True))

print("\nOriginal treatment balance - test:")
print(segment_test.value_counts(normalize=True))

print("\nBinary treatment balance - train:")
print(treatment_train.value_counts(normalize=True))

print("\nBinary treatment balance - test:")
print(treatment_test.value_counts(normalize=True))

print("\nVisit rate - train:", y_train.mean())
print("Visit rate - test:", y_test.mean())

Train size: (48000, 8)
Test size: (16000, 8)

Original treatment balance - train:
segment
Womens E-Mail    0.334167
No E-Mail        0.332917
Mens E-Mail      0.332917
Name: proportion, dtype: float64

Original treatment balance - test:
segment
Womens E-Mail    0.334188
Mens E-Mail      0.332937
No E-Mail        0.332875
Name: proportion, dtype: float64

Binary treatment balance - train:
treatment_binary
1    0.667083
0    0.332917
Name: proportion, dtype: float64

Binary treatment balance - test:
treatment_binary
1    0.667125
0    0.332875
Name: proportion, dtype: float64

Visit rate - train: 0.14677083333333332
Visit rate - test: 0.1468125


### Train/test split balance interpretation

The train/test split appears well balanced. The original three treatment groups are each approximately one-third of the data in both the training and testing sets. After collapsing the treatment into a binary variable, approximately two-thirds of customers are in the email treatment group and one-third are in the no-email control group in both sets. The visit rate is also nearly identical across train and test, which helps ensure that later model comparisons are based on the same stable split.

### Separate categorical and numeric features

This cell separates the raw features into categorical and numeric columns. The categorical columns will be one-hot encoded, while the numeric columns can be used directly.

In [16]:
categorical_cols = [
    "history_segment",
    "zip_code",
    "channel"
]

numeric_cols = [
    "recency",
    "history",
    "mens",
    "womens",
    "newbie"
]

### Fit one-hot encoder on training data

This cell fits the one-hot encoder using only the training data. Fitting the encoder only on the training set avoids using information from the test set during preprocessing.

In [17]:
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

encoder.fit(X_train_raw[categorical_cols])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

### Apply one-hot encoding to train and test data

This cell applies the fitted encoder to both the training and test categorical variables. The same encoder is used for both sets so that the resulting feature columns match exactly.

In [18]:
encoded_train = encoder.transform(X_train_raw[categorical_cols])
encoded_test = encoder.transform(X_test_raw[categorical_cols])

encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

X_train_encoded = pd.DataFrame(
    encoded_train,
    columns=encoded_feature_names,
    index=X_train_raw.index
)

X_test_encoded = pd.DataFrame(
    encoded_test,
    columns=encoded_feature_names,
    index=X_test_raw.index
)

### Build final train and test feature matrices

This cell combines the numeric columns with the one-hot encoded categorical columns. The resulting `X_train` and `X_test` matrices are the final feature matrices that later models should use.

In [20]:
X_train = pd.concat(
    [X_train_raw[numeric_cols], X_train_encoded],
    axis=1
)

X_test = pd.concat(
    [X_test_raw[numeric_cols], X_test_encoded],
    axis=1
)

display(X_train.head())

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

,recency,history,mens,womens,newbie,history_segment_1) $0 - $100,history_segment_2) $100 - $200,history_segment_3) $200 - $350,history_segment_4) $350 - $500,history_segment_5) $500 - $750,"history_segment_6) $750 - $1,000","history_segment_7) $1,000 +",zip_code_Rural,zip_code_Suburban,zip_code_Urban,channel_Multichannel,channel_Phone,channel_Web
17961,9,40.52,0,1,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
49849,8,364.01,0,1,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
10279,5,29.99,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
28814,10,67.09,1,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
16137,3,140.68,0,1,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


X_train shape: (48000, 18)
X_test shape: (16000, 18)


### Validate final feature matrices

This cell performs final checks before saving the processed files. It confirms that train and test columns match, there are no missing values, and the feature, outcome, and treatment arrays have consistent lengths.

In [21]:
assert list(X_train.columns) == list(X_test.columns)
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert len(X_train) == len(y_train) == len(treatment_train)
assert len(X_test) == len(y_test) == len(treatment_test)

print("Validation passed.")

Validation passed.


### Save processed train/test files

This cell saves the final feature matrices, primary outcome, binary treatment labels, secondary outcomes, and original treatment segment labels. These files allow later notebooks to use the exact same train/test split and treatment encoding.

In [22]:
X_train.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(PROCESSED_DIR / "X_test.csv", index=False)

y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

treatment_train.to_csv(PROCESSED_DIR / "treatment_train.csv", index=False)
treatment_test.to_csv(PROCESSED_DIR / "treatment_test.csv", index=False)

secondary_y_train.to_csv(PROCESSED_DIR / "secondary_y_train.csv", index=False)
secondary_y_test.to_csv(PROCESSED_DIR / "secondary_y_test.csv", index=False)

segment_train.to_csv(PROCESSED_DIR / "segment_train.csv", index=False)
segment_test.to_csv(PROCESSED_DIR / "segment_test.csv", index=False)

### Save feature matrix metadata

This cell saves metadata describing the feature columns, treatment encoding, outcome variables, train/test split, and cleaning decisions. This makes the preprocessing choices transparent and reproducible for the team.

In [23]:
metadata = {
    "primary_outcome": primary_outcome,
    "secondary_outcomes": secondary_outcomes,
    "treatment_encoding": {
        "0": "No Email / Control",
        "1": "Email Treatment: Mens E-Mail or Womens E-Mail"
    },
    "original_treatment_column": "segment",
    "binary_treatment_column": "treatment_binary",
    "feature_columns_raw": feature_cols,
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "encoded_feature_columns": list(X_train.columns),
    "test_size": 0.25,
    "random_state": 42,
    "stratification": "segment + visit",
    "cleaning_notes": [
        "Corrected zip_code value 'Surburban' to 'Suburban'.",
        "Excluded post-treatment outcome variables visit, conversion, and spend from model features."
    ]
}

with open(PROCESSED_DIR / "feature_matrix_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

### Confirm saved output files

This cell lists the files saved to the processed data directory and displays the final treatment encoding. This provides a final confirmation that the feature matrix pipeline ran successfully.

In [24]:
print("Saved files to:", PROCESSED_DIR)

print("\nSaved modeling files:")
for file in sorted(PROCESSED_DIR.glob("*")):
    print(file.name)

print("\nTreatment encoding:")
display(
    df[["segment", "treatment_binary", "treatment_label"]]
    .drop_duplicates()
    .sort_values(["treatment_binary", "segment"])
)

print("\nNumber of final model features:", X_train.shape[1])
print("Number of training rows:", X_train.shape[0])
print("Number of testing rows:", X_test.shape[0])

Saved files to: /Users/chris/Library/CloudStorage/OneDrive-Personal/Rice University MDS/COMP 653 Statistical Machine Learning/Uplift Modeling Semester Project/hillstrom-email-uplift-modeling/data/processed

Saved modeling files:
X_test.csv
X_train.csv
feature_matrix_metadata.json
secondary_y_test.csv
secondary_y_train.csv
segment_test.csv
segment_train.csv
treatment_test.csv
treatment_train.csv
y_test.csv
y_train.csv

Treatment encoding:


,segment,treatment_binary,treatment_label
1,No E-Mail,0,No Email
3,Mens E-Mail,1,Email
0,Womens E-Mail,1,Email



Number of final model features: 18
Number of training rows: 48000
Number of testing rows: 16000
